# AI Programming — Lecture 11
## Lab 2-4: MLP for Forward Kinematics

ABB IRB 2400 6-DoF robot arm 데이터로 **Forward Kinematics regression**을 학습합니다.

```text
Joint angles
(q1, q2, q3, q4, q5, q6)
        ↓
       MLP
        ↓
End-effector pose
(x, y, z, yaw, pitch, roll)
```

### 학습 목표
- 다중 입력 / 다중 출력 regression을 구성할 수 있습니다.
- joint angle과 end-effector pose의 관계를 neural network로 학습할 수 있습니다.
- input과 output을 각각 standardize할 수 있습니다.
- train/validation/test set을 분리할 수 있습니다.
- 원래 단위로 prediction을 복원하고 output별 MAE를 계산할 수 있습니다.
- 학습한 model과 scaler를 Google Drive에 저장할 수 있습니다.

### Colab 실행 안내
원 데이터는 약 **300,000 samples**이므로 A100이 아닌 Colab에서 실습하기 쉽게 다음 설정을 사용합니다.

- `batch_size = 512`
- `epochs = 50`
- `EarlyStopping(patience=7)`
- 기본 Colab GPU(T4 등) 권장
- CPU에서도 실행 가능하지만 시간이 더 걸릴 수 있습니다.

데이터 파일:

```text
MyDrive/Colab Notebooks/data/datasetIRB2400.csv
```

## 1. 라이브러리와 Colab 환경 설정

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 50
PATIENCE = 7

tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow version:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "사용하지 않음 (CPU)")

## 2. Google Drive 마운트와 데이터 불러오기

ABB IRB 2400 dataset에는 joint angle과 end-effector pose가 함께 저장되어 있습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

csv_path = (
    '/content/drive/MyDrive/Colab Notebooks/data/'
    'datasetIRB2400.csv'
)

df = pd.read_csv(csv_path)

print("데이터셋 로딩 완료")
print("Shape:", df.shape)
display(df.head())

## 3. Input과 Target 설정

Forward kinematics에서는

```text
Input  : q1_in ~ q6_in
Target : x, y, z, yaw, pitch, roll
```

을 사용합니다.

- `x, y, z`: end-effector position
- `yaw, pitch, roll`: end-effector orientation

In [ ]:
X = df[
    ['q1_in', 'q2_in', 'q3_in', 'q4_in', 'q5_in', 'q6_in']
].values

y = df[
    ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
].values

print("X shape:", X.shape)
print("y shape:", y.shape)

## 4. Train / Validation / Test 분할

원본 notebook과 동일하게 먼저 20%를 test로 분리하고,
남은 80% 중 20%를 validation으로 사용합니다.

최종 비율:

```text
Train      64%
Validation 16%
Test       20%
```

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=SEED,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 5. Input / Output Standardization

Scaler는 반드시 **train set에서만 fit**합니다.

Forward kinematics의 output은 위치와 각도가 서로 다른 scale을 가지므로,
target도 standardization한 뒤 학습하는 것이 안정적입니다.

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)
y_test_scaled = scaler_y.transform(y_test)

print("Scaled input mean:", X_train_scaled.mean(axis=0))
print("Scaled target mean:", y_train_scaled.mean(axis=0))

## 6. MLP Model

구조:

```text
6
→ Dense(128, ReLU)
→ Dense(128, ReLU)
→ Dense(64, ReLU)
→ Dense(6)
```

Regression이므로 마지막 layer에는 activation을 사용하지 않습니다.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(6,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(6),
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

model.summary()

## 7. Early Stopping과 Model Training

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_data=(X_val_scaled, y_val_scaled),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1,
)

## 8. Learning Curve 확인

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Forward Kinematics: Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## 9. Test Prediction과 원래 단위 복원

Neural network는 standardized target을 예측합니다.
따라서 평가하기 전에 `inverse_transform()`으로 원래 단위로 복원합니다.

In [ ]:
test_loss, test_mae = model.evaluate(
    X_test_scaled,
    y_test_scaled,
    verbose=0,
)

print(f"Test Loss (scaled MSE): {test_loss:.4f}")
print(f"Test MAE  (scaled): {test_mae:.4f}")

y_pred_scaled = model.predict(
    X_test_scaled,
    batch_size=BATCH_SIZE,
    verbose=0,
)

y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = y_test

mae_real = mean_absolute_error(y_true, y_pred)
rmse_real = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Overall MAE  : {mae_real:.4f}")
print(f"Overall RMSE : {rmse_real:.4f}")
print(f"R2 Score     : {r2:.4f}")

## 10. Output별 MAE

위치 `x, y, z`와 자세 `yaw, pitch, roll`은 단위와 scale이 다릅니다.

따라서 overall MAE 하나만 보기보다 **각 output별 MAE를 반드시 함께 확인**해야 합니다.

In [ ]:
output_labels = ["x", "y", "z", "yaw", "pitch", "roll"]

for i, label in enumerate(output_labels):
    mae_i = mean_absolute_error(
        y_true[:, i],
        y_pred[:, i],
    )
    print(f"MAE for {label:>5}: {mae_i:.4f}")

## 11. Prediction 시각화

In [ ]:
n_plot = 300

plt.figure(figsize=(10, 5))
plt.plot(
    y_true[:n_plot, 0],
    label="True x",
)
plt.plot(
    y_pred[:n_plot, 0],
    label="Predicted x",
)
plt.xlabel("Test Sample Index")
plt.ylabel("x")
plt.title("Forward Kinematics: True vs. Predicted x")
plt.legend()
plt.grid(True)
plt.show()

## 12. Model과 Scaler 저장

In [ ]:
save_dir = (
    '/content/drive/MyDrive/Colab Notebooks/models'
)
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(
    save_dir,
    'mlp_forward_kinematics.keras'
)
scaler_X_path = os.path.join(
    save_dir,
    'scaler_X_forward_kinematics.pkl'
)
scaler_y_path = os.path.join(
    save_dir,
    'scaler_y_forward_kinematics.pkl'
)

model.save(model_path)
joblib.dump(scaler_X, scaler_X_path)
joblib.dump(scaler_y, scaler_y_path)

print("모델 저장:", model_path)
print("입력 scaler 저장:", scaler_X_path)
print("출력 scaler 저장:", scaler_y_path)

## 13. 저장한 Model로 한 Sample 예측

In [ ]:
loaded_model = keras.models.load_model(model_path)
loaded_scaler_X = joblib.load(scaler_X_path)
loaded_scaler_y = joblib.load(scaler_y_path)

sample_q = X_test[0:1]

sample_q_scaled = loaded_scaler_X.transform(sample_q)
sample_pose_scaled = loaded_model.predict(
    sample_q_scaled,
    verbose=0,
)
sample_pose = loaded_scaler_y.inverse_transform(
    sample_pose_scaled
)

print("입력 관절각:")
print(sample_q)

print("\n예측된 말단 위치/자세:")
print(sample_pose)

print("\n실제 말단 위치/자세:")
print(y_test[0:1])

## 14. 직접 해보기

1. `batch_size=256`과 `512`의 학습 시간을 비교하세요.
2. hidden unit를 `128 → 128 → 64`에서 `256 → 128 → 64`로 바꾸어 보세요.
3. 위치와 자세 중 어느 output의 MAE가 더 큰지 확인하세요.
4. `EarlyStopping`이 실제로 몇 epoch에서 종료되었는지 확인하세요.
5. position과 orientation에 서로 다른 loss weight를 주는 방법을 생각해 보세요.

### 체크포인트

Forward kinematics는

```text
joint angles → pose
```

의 mapping을 학습하는 **다중 출력 regression**입니다.